# CineIQ — Weighted EnsembleCombines content-based (top-K genome), SVD collaborative filtering, and popularity via weighted sum.Uses shared functions from `src/recommender.py`.

In [ ]:
import sys
sys.path.append("..")

import pandas as pd
import numpy as np
import json

from src.recommender import (
    load_movies, load_ratings, load_content_index,
    load_svd_model, load_weights, load_popularity,
    content_scores, svd_scores, popularity_scores, ensemble_recommend
)

movies = load_movies()
ratings = load_ratings()
content_index = load_content_index()
svd_model = load_svd_model()
weights = load_weights()
popularity = load_popularity(ratings)

print(f"Movies: {len(movies):,}")
print(f"Content index: {len(content_index):,} movies")
print(f"Weights: {weights}")

## Test Ensemble

In [ ]:
result = ensemble_recommend(
    user_id=1,
    liked_movie_title="Toy Story (1995)",
    content_index=content_index,
    movies=movies,
    ratings=ratings,
    svd_model=svd_model,
    popularity=popularity,
    weights=weights,
    n=10
)
print("=== Top 10 Recommendations for User 1 (liked Toy Story) ===")
result[["title", "genres", "score"]]

In [ ]:
# Test with another movie
result2 = ensemble_recommend(
    user_id=42,
    liked_movie_title="Matrix, The (1999)",
    content_index=content_index,
    movies=movies,
    ratings=ratings,
    svd_model=svd_model,
    popularity=popularity,
    weights=weights,
    n=10
)
print("=== Top 10 for User 42 (liked The Matrix) ===")
result2[["title", "genres", "score"]]

## Validate Ensemble WeightsTest different weight combinations to find optimal.

In [ ]:
# Grid search over weight combinations
best_weights = None
best_score = -1

for w_content in np.arange(0.1, 0.6, 0.1):
    for w_svd in np.arange(0.2, 0.7, 0.1):
        w_pop = round(1.0 - w_content - w_svd, 2)
        if w_pop < 0:
            continue
        test_weights = {"w_content": round(w_content, 2), "w_svd": round(w_svd, 2), "w_pop": w_pop}

        result = ensemble_recommend(
            user_id=1, liked_movie_title="Toy Story (1995)",
            content_index=content_index, movies=movies, ratings=ratings,
            svd_model=svd_model, popularity=popularity,
            weights=test_weights, n=10
        )
        if result is not None:
            avg_score = result["score"].mean()
            if avg_score > best_score:
                best_score = avg_score
                best_weights = test_weights

print(f"Best weights: {best_weights}")
print(f"Best avg score: {best_score:.4f}")

In [ ]:
# Save best weights
with open("../models/ensemble_weights.json", "w") as f:
    json.dump(best_weights, f, indent=2)
print(f"Saved ensemble_weights.json: {best_weights}")